<a href="https://colab.research.google.com/github/nataliamarinn/labo3-2026r/blob/main/src/AutoGluon/z329_ErrorIrreducible.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Error Irreducible — ¿Qué parte del error no se puede corregir?

## El problema

En el backtesting vimos que para algunos productos **todos los modelos se equivocan en la misma dirección**:

```
product 20008: real=195  | naive=469  har=444  arima=419  ag=390
product 20014: real=272  | naive=479  har=434  arima=455  ag=390
```

Eso no es ruido aleatorio. Cuando todos sobre-predicen por el mismo factor (~2x),
hay un **quiebre estructural**: el producto cayó a la mitad y ningún modelo lo anticipó.

## Dos tipos de error

| Tipo | Causa | ¿Corregible? |
|---|---|---|
| **Ruido aleatorio** | La serie siempre fue volátil | No — es varianza inherente |
| **Quiebre estructural** | El proceso generador cambió de golpe | No desde la historia sola |
| **Error de modelo** | El modelo no captura la estructura que sí existe | Sí |

## Cuadrantes de diagnóstico

```
                    Error grande    Error chico
                 ┌──────────────┬─────────────┐
Serie            │  QUIEBRE     │  BIEN       │
estructurada     │  ESTRUCTURAL │  MODELADO   │
(R² alto)        │  irreducible │             │
                 ├──────────────┼─────────────┤
Serie ruidosa    │  RUIDO       │  RUIDO      │
(R² bajo)        │  QUIEBRE     │  NORMAL     │
                 └──────────────┴─────────────┘
```

## Este notebook es auto-contenido
No necesita ningún CSV externo. Recalcula Naive + HAR + Ridge Panel directamente.

## 0.1 Init ambiente Google Colab

In [ ]:
from google.colab import drive
drive.mount('/content/.drive')

In [ ]:
%%shell

mkdir -p "/content/.drive/My Drive/labo3"
mkdir -p "/content/buckets"
ln -sfn "/content/.drive/My Drive/labo3"   /content/buckets/b1

mkdir -p ~/.kaggle
cp /content/buckets/b1/kaggle/kaggle.json  ~/.kaggle
chmod 600 ~/.kaggle/kaggle.json

mkdir -p /content/buckets/b1/exp
mkdir -p /content/buckets/b1/datasets
mkdir -p /content/datasets

descargar() {
  carpeta_destino="/content/buckets/b1/datasets/"
  url_origen="https://storage.googleapis.com/open-courses/austral2026-5da5/labo3/"
  archivo="$1"

  if ! test -f "$carpeta_destino""$archivo"; then
    wget  "$url_origen""$archivo"  -O "$carpeta_destino""$archivo"
  fi

  if ! test -f  "/content/datasets/""$archivo"; then
    cp  "$carpeta_destino""$archivo"  "/content/datasets/""$archivo"
  fi;
}

descargar  "sell-in.txt.gz"
descargar  "tb_productos.txt"
descargar  "tb_stocks.txt"
descargar  "product_id_apredecir201912.txt"

# 1  Setup

In [ ]:
!pip install uv
!uv pip install -q kaggle

In [ ]:
import os
import numpy as np
import polars as pl
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression, RidgeCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score

import warnings
warnings.filterwarnings('ignore')

In [ ]:
PARAM = {
  'experimento':    'ErrorIrreducible-01',
  'semilla':        102191,
  'periodo_corte':  201910,   # entrenamos hasta acá
  'periodo_target': 201912,   # predecimos esto (t+2, valor real conocido)
  'sigma_quiebre':  2.0,      # umbral de sorpresa en sigmas
  'r2_estructura':  0.3,      # R² mínimo para considerar serie "estructurada"
}

In [ ]:
ruta = "/content/buckets/b1/exp/" + PARAM['experimento']
os.makedirs(ruta, exist_ok=True)
os.chdir(ruta)

# 2  Datos

In [ ]:
dataset = pl.read_csv('/content/.drive/My Drive/labo3/datasets/sell-in.txt.gz', separator="\t")

tb_ventas = dataset.group_by("product_id", "periodo").agg(
    pl.col("tn").sum().alias("tn")
).sort(["product_id", "periodo"])

tb_apredecir = pl.read_csv('/content/.drive/My Drive/labo3/datasets/product_id_apredecir201912.txt', separator="\t")
tb_ventas    = tb_ventas.join(tb_apredecir, on="product_id", how="inner").sort(["product_id", "periodo"])

tb_train = tb_ventas.filter(pl.col("periodo") <= PARAM['periodo_corte'])
tb_real  = (
    tb_ventas
    .filter(pl.col("periodo") == PARAM['periodo_target'])
    .select(["product_id", "tn"])
    .rename({"tn": "tn_real"})
)

productos = tb_apredecir["product_id"].to_list()
print(f"Train: hasta {PARAM['periodo_corte']}  |  {tb_train.height} filas")
print(f"Real : {PARAM['periodo_target']}       |  {tb_real.height} productos")

# 3  Predicciones de los 3 modelos (inline, sin CSV externo)

Calculamos directamente sobre `tb_train`:
- **Naive** (mediana 6m) — baseline
- **HAR** (HAR lineal + Racha filter) — captura estructura
- **Ridge Panel** (features de lags + predicción directa a t+2) — captura patrones entre productos

Los tres corren en menos de 2 minutos.

In [ ]:
# ── helpers ────────────────────────────────────────────────────────────────

def build_har_features(serie):
    T = len(serie)
    X, y = [], []
    for t in range(12, T):
        X.append([serie[t-1], serie[t-3:t].mean(), serie[t-6:t].mean(), serie[t-12:t].mean()])
        y.append(serie[t])
    return np.array(X), np.array(y)

def har_predict_next(m, serie):
    t = len(serie)
    x = np.array([[serie[t-1], serie[t-3:t].mean(), serie[t-6:t].mean(), serie[t-12:t].mean()]])
    return float(m.predict(x)[0])

def safe_mean(a):
    return float(a.mean()) if len(a) > 0 else 0.0

In [ ]:
# ── Naive: mediana 6m ──────────────────────────────────────────────────────
preds_naive = []
for pid in productos:
    serie = tb_train.filter(pl.col("product_id") == pid).sort("periodo")["tn"].to_numpy().astype(float)
    w6    = serie[max(0, len(serie)-6):]
    preds_naive.append({'product_id': pid, 'pred_naive': max(float(np.median(w6)), 0.0)})

tb_naive = pl.DataFrame(preds_naive)
print("Naive listo")

In [ ]:
# ── HAR + Test de Racha ────────────────────────────────────────────────────
from scipy.stats import runstest_1samp

preds_har = []
for pid in productos:
    serie    = tb_train.filter(pl.col("product_id") == pid).sort("periodo")["tn"].to_numpy().astype(float)
    fallback = float(serie[-12:].mean()) if len(serie) >= 12 else float(serie.mean())
    try:
        _, pv = runstest_1samp(serie, cutoff='median')
        tiene_estructura = pv < 0.05
    except Exception:
        tiene_estructura = False

    if not tiene_estructura or len(serie) < 14:
        pred = fallback
    else:
        try:
            X, y = build_har_features(serie)
            m    = LinearRegression().fit(X, y)
            p1   = max(har_predict_next(m, serie), 0.0)
            pred = max(har_predict_next(m, np.append(serie, p1)), 0.0)
        except Exception:
            pred = fallback

    preds_har.append({'product_id': pid, 'pred_har': pred})

tb_har = pl.DataFrame(preds_har)
print("HAR listo")

In [ ]:
# ── Ridge Panel (directo a t+2) ────────────────────────────────────────────
# construye la tabla panel de TODOS los productos sobre tb_train
filas_panel = []
for pid in productos:
    serie_df  = tb_train.filter(pl.col("product_id") == pid).sort("periodo")
    periodos_ = serie_df["periodo"].to_list()
    tn_       = serie_df["tn"].to_numpy().astype(float)
    T         = len(tn_)
    for t in range(12, T - 2):
        mes_t2 = int(str(periodos_[t + 2])[4:6])
        filas_panel.append([
            pid, periodos_[t],
            tn_[t-1], tn_[t-2], tn_[t-3],
            safe_mean(tn_[t-3:t]), safe_mean(tn_[t-6:t]), safe_mean(tn_[t-12:t]),
            mes_t2, tn_[t+2]
        ])

cols_panel = ['product_id','periodo_t','lag1','lag2','lag3','mean_3m','mean_6m','mean_12m','mes','tn_t2']
tb_panel   = pl.DataFrame(filas_panel, schema=cols_panel)

FEAT = ['lag1','lag2','lag3','mean_3m','mean_6m','mean_12m','mes','product_id']
X_all = tb_panel.select(FEAT).to_numpy()
y_all = tb_panel['tn_t2'].to_numpy()

sc = StandardScaler()
ridge = RidgeCV(alphas=[0.1, 1.0, 10.0, 100.0], cv=5)
ridge.fit(sc.fit_transform(X_all), y_all)

# predicción para 201912 (t+2 desde 201910)
preds_ridge = []
for pid in productos:
    serie = tb_train.filter(pl.col("product_id") == pid).sort("periodo")["tn"].to_numpy().astype(float)
    t     = len(serie) - 1
    fila  = np.array([[
        serie[t-1] if t >= 1 else 0.0,
        serie[t-2] if t >= 2 else 0.0,
        serie[t-3] if t >= 3 else 0.0,
        safe_mean(serie[max(0,t-3):t]),
        safe_mean(serie[max(0,t-6):t]),
        safe_mean(serie[max(0,t-12):t]),
        12,   # mes = diciembre (201912)
        pid
    ]])
    pred = max(float(ridge.predict(sc.transform(fila))[0]), 0.0)
    preds_ridge.append({'product_id': pid, 'pred_ridge': pred})

tb_ridge = pl.DataFrame(preds_ridge)
print("Ridge Panel listo")

In [ ]:
# ── Tabla de errores ───────────────────────────────────────────────────────
tb_errores = (
    tb_real
    .join(tb_naive,  on='product_id', how='left')
    .join(tb_har,    on='product_id', how='left')
    .join(tb_ridge,  on='product_id', how='left')
)

MODELOS = ['naive', 'har', 'ridge']
for m in MODELOS:
    tb_errores = tb_errores.with_columns(
        (pl.col('tn_real') - pl.col(f'pred_{m}')).abs().alias(f'err_{m}')
    )
tb_errores = tb_errores.with_columns(
    ((pl.col('err_naive') + pl.col('err_har') + pl.col('err_ridge')) / 3).alias('err_promedio')
)

print("RMSE por modelo (backtesting sobre 201912):")
for m in MODELOS:
    rmse = float(np.sqrt((tb_errores[f'err_{m}'] ** 2).mean()))
    print(f"  {m:8s}: {rmse:.4f}")

# 4  Diagnóstico por producto

| Métrica | Qué mide |
|---|---|
| `r2_har` | R² HAR in-sample — ¿cuánta estructura predecible tiene la serie? |
| `cv` | Volatilidad relativa histórica |
| `sorpresa` | Cuántas sigmas se alejó 201912 de la media histórica |
| `nivel_caida` | tn_real / media_train (< 0.5 = cayó a la mitad) |
| `direccion` | Todos sobre-predicen (+) / sub-predicen (-) / mixto (~0) |
| `consenso` | Bajo = todos se equivocan igual → quiebre claro |

In [ ]:
diagnostico = []

for pid in productos:
    serie       = tb_train.filter(pl.col("product_id") == pid).sort("periodo")["tn"].to_numpy().astype(float)
    real        = float(tb_real.filter(pl.col("product_id") == pid)["tn_real"][0])
    row_err     = tb_errores.filter(pl.col("product_id") == pid)

    media_train = serie.mean()
    std_train   = serie.std() + 1e-9

    # R² HAR in-sample
    r2 = np.nan
    if len(serie) >= 14:
        try:
            Xh, yh = build_har_features(serie)
            r2 = float(r2_score(yh, LinearRegression().fit(Xh, yh).predict(Xh)))
        except Exception:
            pass

    sorpresa    = (real - media_train) / std_train
    nivel_caida = real / (media_train + 1e-9)

    preds_vals  = np.array([
        float(row_err['pred_naive'][0]),
        float(row_err['pred_har'][0]),
        float(row_err['pred_ridge'][0]),
    ])
    errores_sig = preds_vals - real
    direccion   = float(np.mean(errores_sig))
    consenso    = float(np.std(errores_sig) / (np.mean(np.abs(errores_sig)) + 1e-9))

    diagnostico.append({
        'product_id':   pid,
        'tn_real':      real,
        'media_train':  float(media_train),
        'std_train':    float(std_train),
        'cv':           float(std_train / (media_train + 1e-9)),
        'r2_har':       r2,
        'sorpresa':     float(sorpresa),
        'nivel_caida':  float(nivel_caida),
        'direccion':    direccion,
        'consenso':     float(consenso),
        'err_promedio': float(row_err['err_promedio'][0]),
    })

tb_diag = pl.DataFrame(diagnostico)
print(f"Diagnóstico completo: {tb_diag.height} productos")
display(tb_diag.sort('err_promedio', descending=True).head(10))

# 5  Clasificación en cuadrantes

In [ ]:
tb_diag = tb_diag.with_columns([
    (pl.col('r2_har') >= PARAM['r2_estructura']).alias('es_estructurado'),
    (pl.col('sorpresa').abs() >= PARAM['sigma_quiebre']).alias('es_quiebre'),
])

def etiquetar(es_struct, es_quiebre):
    if es_struct and es_quiebre:      return 'QUIEBRE_ESTRUCTURAL'
    if es_struct and not es_quiebre:  return 'BIEN_MODELADO'
    if not es_struct and es_quiebre:  return 'RUIDO_QUIEBRE'
    return 'RUIDO_NORMAL'

tb_diag = tb_diag.with_columns(
    pl.Series('categoria', [etiquetar(bool(r['es_estructurado']), bool(r['es_quiebre']))
                             for r in tb_diag.iter_rows(named=True)])
)

err_total_global = tb_diag['err_promedio'].sum()
cats_orden   = ['QUIEBRE_ESTRUCTURAL', 'RUIDO_QUIEBRE', 'RUIDO_NORMAL', 'BIEN_MODELADO']
colores_cat  = {'QUIEBRE_ESTRUCTURAL': 'tomato', 'BIEN_MODELADO': 'steelblue',
                'RUIDO_QUIEBRE': 'darkorange', 'RUIDO_NORMAL': 'lightgray'}

print("\n% del error total por categoría:")
for cat in cats_orden:
    sub = tb_diag.filter(pl.col('categoria') == cat)
    n   = sub.height
    pct = sub['err_promedio'].sum() / err_total_global * 100
    print(f"  {cat:25s}: {n:4d} productos ({n/len(productos)*100:.1f}%)  →  {pct:.1f}% del error total")

# 6  Visualización: cuadrante R² vs Sorpresa

Cada punto es un producto. Tamaño ∝ error promedio.

In [ ]:
df_plot = tb_diag.to_pandas().dropna(subset=['r2_har'])

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# izquierda: cuadrante principal
for cat, color in colores_cat.items():
    sub  = df_plot[df_plot['categoria'] == cat]
    size = np.clip(sub['err_promedio'] * 2, 10, 300)
    axes[0].scatter(sub['sorpresa'], sub['r2_har'], c=color, s=size,
                    alpha=0.6, label=cat, edgecolors='white', linewidth=0.3)

axes[0].axvline(-PARAM['sigma_quiebre'], color='red', linestyle='--', linewidth=0.8,
                label=f'±{PARAM["sigma_quiebre"]}σ')
axes[0].axvline( PARAM['sigma_quiebre'], color='red', linestyle='--', linewidth=0.8)
axes[0].axhline(PARAM['r2_estructura'], color='navy', linestyle='--', linewidth=0.8,
                label=f'R²={PARAM["r2_estructura"]}')
axes[0].set_xlabel('Sorpresa (sigmas respecto a media histórica)', fontsize=9)
axes[0].set_ylabel('R² HAR in-sample', fontsize=9)
axes[0].set_title('Diagnóstico por producto\n(tamaño = error promedio)', fontsize=9)
axes[0].legend(fontsize=7)
axes[0].text(-5.5, 0.88, 'QUIEBRE\nESTRUCTURAL', ha='center', fontsize=7,
             color='tomato', fontweight='bold')
axes[0].text( 0.0, 0.88, 'BIEN\nMODELADO',      ha='center', fontsize=7, color='steelblue')
axes[0].text( 0.0, 0.05, 'RUIDO NORMAL',          ha='center', fontsize=7, color='gray')

# derecha: nivel de caída vs error
for cat, color in colores_cat.items():
    sub = df_plot[df_plot['categoria'] == cat]
    axes[1].scatter(sub['nivel_caida'], sub['err_promedio'], c=color, s=25,
                    alpha=0.6, label=cat, edgecolors='white', linewidth=0.3)

axes[1].axvline(1.0, color='black', linestyle='--', linewidth=0.8, label='nivel sin cambio')
axes[1].axvline(0.5, color='red',   linestyle=':',  linewidth=0.8, label='cayó al 50%')
axes[1].set_xlabel('tn_real / media_train  (1=sin cambio, <0.5=cayó a la mitad)', fontsize=9)
axes[1].set_ylabel('Error promedio entre modelos', fontsize=9)
axes[1].set_title('Caída de nivel vs error', fontsize=9)
axes[1].set_xlim(left=0)
axes[1].legend(fontsize=7)

plt.tight_layout()
plt.show()

# 7  ¿Cuánto del error es irreducible?

Descomponemos el SSE total en la contribución de cada categoría.

In [ ]:
tb_full = tb_diag.join(
    tb_errores.select(['product_id','err_naive','err_har','err_ridge']),
    on='product_id', how='left'
)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
colores_bar = [colores_cat[c] for c in cats_orden]

# RMSE por categoría × modelo
x = np.arange(3)
width = 0.2
for i, (cat, color) in enumerate(zip(cats_orden, colores_bar)):
    sub  = tb_full.filter(pl.col('categoria') == cat)
    vals = [float(np.sqrt((sub[f'err_{m}'].to_numpy()**2).mean())) if sub.height > 0 else 0
            for m in ['naive', 'har', 'ridge']]
    axes[0].bar(x + i * width, vals, width, label=cat, color=color, alpha=0.85)

axes[0].set_xticks(x + width * 1.5)
axes[0].set_xticklabels(['Naive', 'HAR', 'Ridge'])
axes[0].set_ylabel('RMSE')
axes[0].set_title('RMSE por modelo y categoría')
axes[0].legend(fontsize=7)

# % SSE por categoría (usando HAR como referencia)
sse_total = (tb_full['err_har'].to_numpy()**2).sum()
pcts = []
for cat in cats_orden:
    sub = tb_full.filter(pl.col('categoria') == cat)
    pcts.append((sub['err_har'].to_numpy()**2).sum() / sse_total * 100)

axes[1].barh(cats_orden, pcts, color=colores_bar, alpha=0.85)
for i, v in enumerate(pcts):
    axes[1].text(v + 0.3, i, f'{v:.1f}%', va='center', fontsize=9)
axes[1].set_xlabel('% del SSE total (HAR)')
axes[1].set_title('¿Cuánto del error total viene de cada categoría?')

plt.tight_layout()
plt.show()

q_pct = pcts[cats_orden.index('QUIEBRE_ESTRUCTURAL')]
b_pct = pcts[cats_orden.index('BIEN_MODELADO')]
print(f"\n→ Error irreducible (quiebre estructural): {q_pct:.1f}% del SSE")
print(f"→ Error mejorable  (bien modelado):        {b_pct:.1f}% del SSE")

# 8  Series con quiebre — ¿era visible antes?

Calculamos la **pendiente de los últimos 6 meses** antes del corte.
Si la tendencia ya era negativa → la caída tenía señal y era parcialmente evitable.
Si era plana y la caída fue abrupta → genuinamente irreducible.

In [ ]:
pendientes = []
for pid in productos:
    serie = tb_train.filter(pl.col("product_id") == pid).sort("periodo")["tn"].to_numpy().astype(float)
    if len(serie) >= 6:
        ult6       = serie[-6:]
        slope_norm = float(np.polyfit(np.arange(6), ult6, 1)[0]) / (serie.mean() + 1e-9)
    else:
        slope_norm = 0.0
    pendientes.append({'product_id': pid, 'pendiente_norm_6m': slope_norm})

tb_diag = tb_diag.join(pl.DataFrame(pendientes), on='product_id', how='left')

sub_q     = tb_diag.filter(pl.col('categoria') == 'QUIEBRE_ESTRUCTURAL').to_pandas()
sub_resto = tb_diag.filter(pl.col('categoria') != 'QUIEBRE_ESTRUCTURAL').to_pandas()

fig, ax = plt.subplots(figsize=(10, 5))
ax.scatter(sub_resto['pendiente_norm_6m'], sub_resto['sorpresa'],
           c='lightgray', s=15, alpha=0.5, label='otros')
sc = ax.scatter(sub_q['pendiente_norm_6m'], sub_q['sorpresa'],
                c=sub_q['err_promedio'], cmap='Reds', s=70, alpha=0.85,
                label='quiebre estructural', edgecolors='black', linewidth=0.4)
plt.colorbar(sc, label='error promedio')

ax.axhline(-PARAM['sigma_quiebre'], color='red',   linestyle='--', linewidth=0.8)
ax.axvline(0,                       color='black', linestyle='--', linewidth=0.5)
ax.axvline(-0.05,                   color='orange',linestyle=':',  linewidth=0.8,
           label='pendiente -5%/mes')
ax.set_xlabel('Pendiente normalizada últimos 6m de train  (negativa = caída previa)', fontsize=9)
ax.set_ylabel('Sorpresa en 201912 (sigmas)', fontsize=9)
ax.set_title('¿La caída ya era visible antes del quiebre?\n'
             'Izquierda del eje = caída en curso  |  Centro/derecha = ruptura abrupta', fontsize=9)
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

n_gradual = (sub_q['pendiente_norm_6m'] < -0.05).sum()
n_abrupto = (sub_q['pendiente_norm_6m'] >= -0.05).sum()
print(f"Quiebres con caída ya visible (pendiente < -5%/mes): {n_gradual}  → parcialmente evitables")
print(f"Quiebres abruptos (sin señal previa):                {n_abrupto}  → genuinamente irreducibles")

# 9  Visualización de las series con quiebre

Mostramos los 9 productos con mayor error en la categoría `QUIEBRE_ESTRUCTURAL`.
La línea roja punteada es la tendencia de los últimos 6 meses de train.
Los diamantes son las predicciones de cada modelo; el círculo negro es el valor real.

In [ ]:
pids_quiebre = (
    tb_diag.filter(pl.col('categoria') == 'QUIEBRE_ESTRUCTURAL')
    .sort('err_promedio', descending=True).head(9)['product_id'].to_list()
)

ncols = 3
nrows = (len(pids_quiebre) + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(15, 4 * nrows))
axes = axes.flatten()

for i, pid in enumerate(pids_quiebre):
    serie_full = tb_ventas.filter(pl.col('product_id') == pid).sort('periodo')
    periodos_  = serie_full['periodo'].to_list()
    tn_        = serie_full['tn'].to_numpy().astype(float)

    idx_corte  = next((j for j, p in enumerate(periodos_) if p > PARAM['periodo_corte']), len(periodos_))
    idx_target = next((j for j, p in enumerate(periodos_) if p == PARAM['periodo_target']), None)
    tn_train_  = tn_[:idx_corte]

    # tendencia últimos 6 meses
    ult6_  = tn_train_[-6:]
    x6_    = np.arange(len(tn_train_) - 6, len(tn_train_))
    coef_  = np.polyfit(x6_, ult6_, 1)
    slope_ = coef_[0]

    row_err  = tb_errores.filter(pl.col('product_id') == pid)
    sorpresa = float(tb_diag.filter(pl.col('product_id') == pid)['sorpresa'][0])

    ax = axes[i]
    ax.plot(range(len(tn_train_)), tn_train_, 'o-', color='steelblue',
            markersize=3, linewidth=1.5, label='historia')

    x6_ext = np.array([len(tn_train_) - 6, len(tn_train_) + 1])
    ax.plot(x6_ext, np.polyval(coef_, x6_ext), 'r--', linewidth=1.3, alpha=0.8,
            label=f'tend.6m ({slope_:+.1f}/mes)')

    if idx_target is not None:
        ax.scatter([idx_target], [tn_[idx_target]], color='black', s=90, zorder=6,
                   label=f'real={tn_[idx_target]:.1f}')
        for m_lbl, m_col, m_color in [
            ('naive', 'pred_naive', 'gray'),
            ('HAR',   'pred_har',   'green'),
            ('Ridge', 'pred_ridge', 'red')
        ]:
            ax.scatter([idx_target], [float(row_err[m_col][0])],
                       color=m_color, s=50, alpha=0.8, marker='D', label=f'{m_lbl}={float(row_err[m_col][0]):.1f}')

    ax.set_title(f'pid {pid}  |  sorpresa={sorpresa:.1f}σ', fontsize=8)
    ax.legend(fontsize=5.5)

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

fig.suptitle('Top quiebres estructurales — historia + tendencia + preds vs real 201912', fontsize=10)
plt.tight_layout()
plt.show()

# 10  Guardar y conclusión

In [ ]:
tb_resumen = tb_diag.select([
    'product_id', 'categoria', 'tn_real', 'media_train',
    'r2_har', 'cv', 'sorpresa', 'nivel_caida',
    'pendiente_norm_6m', 'direccion', 'consenso', 'err_promedio'
]).sort('err_promedio', descending=True)

tb_resumen.write_csv('diagnostico_error_irreducible.csv')

import shutil
shutil.copy('diagnostico_error_irreducible.csv',
            '/content/.drive/My Drive/labo3/exp/diagnostico_error_irreducible.csv')
print("Guardado local y en Drive")

print("\n══════════════════════════════════════")
print("CONCLUSIÓN")
print("══════════════════════════════════════")
total_prods = tb_diag.height
for cat in cats_orden:
    sub     = tb_diag.filter(pl.col('categoria') == cat)
    n       = sub.height
    err_med = sub['err_promedio'].mean()
    pct_n   = n / total_prods * 100
    pct_err = sub['err_promedio'].sum() / err_total_global * 100
    print(f"  {cat:25s}: {n:4d} prods ({pct_n:.1f}%)  error medio={err_med:.2f}  → {pct_err:.1f}% del error total")

n_ab = (sub_q['pendiente_norm_6m'] >= -0.05).sum()
n_gr = (sub_q['pendiente_norm_6m'] <  -0.05).sum()
print(f"\nDe los quiebres estructurales:")
print(f"  {n_ab} abruptos (genuinamente irreducibles — sin señal previa)")
print(f"  {n_gr} graduales (parcialmente evitables — caída ya en curso)")